In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.read_csv('vuelos_10k.csv')


# Fase 1:

In [ ]:
#Valores nulos
df = df.dropna(subset=['Origen', 'Destino'])

#Formato inconsistente
df['Destino'] = df['Destino'].apply(lambda x: x.capitalize())
df['Origen'] = df['Origen'].apply(lambda x: x.capitalize())

#Anomalias logicas
df = df[df['Origen'] != df['Destino']]

#Errores de captura (distancia negativa)
df['Distancia_km'] = df['Distancia_km'].abs()

#Duplicados (rutas unicas)
df = df.drop_duplicates(subset=['Origen', 'Destino'])


# Fase 2:

Importen la librería networkx.

Utilicen la función nx.from_pandas_edgelist() para convertir su DataFrame en un Grafo Dirigido (nx.DiGraph).

Impriman en pantalla cuántos Nodos (Aeropuertos) y cuántas Aristas (Rutas) tiene su grafo final.

In [ ]:
!pip install networkx

In [ ]:
G = nx.from_pandas_edgelist(df, source='Origen', target='Destino', edge_attr=None, create_using=nx.DiGraph)

In [ ]:
print(f"Nodos (Aeropuertos): {G.number_of_nodes()}")
print(f"Aristas (Rutas): {G.number_of_edges()}")

Nodos (Aeropuertos): 3253
Aristas (Rutas): 9743


# Fase 3:

¡Llegó la hora de comprobar la teoría de la clase!

- Construyan la Matriz de Adyacencia densa a partir de su grafo usando nx.adjacency_matrix(G).todense().
Utilicen código Python (ej. el atributo .nbytes) para calcular cuántos Megabytes (MB) de Memoria RAM está consumiendo su Matriz de Adyacencia llena de ceros.
- Calculen cuántos Megabytes está consumiendo su DataFrame limpio (la Lista de Aristas).
- Veredicto: Añadan una celda Markdown al final de su notebook respondiendo: ¿Cuántas veces más pesada es la Matriz en comparación con la Lista de Aristas y por qué sucede esto?

In [ ]:
A = nx.adjacency_matrix(G).todense()
A = np.array(A)

In [ ]:
matriz_bytes = A.nbytes
matriz_mb = matriz_bytes / (1024 * 1024)

df_bytes = df.memory_usage(deep=True).sum()
df_mb = df_bytes / (1024 * 1024)

print(f"Consumo de Memoria RAM de la Matriz de Adyacencia densa: {matriz_mb:.4f} MB")
print(f"Consumo de Memoria RAM del Dataframe limpio (lista de aristas): {df_mb:.4f} MB")

Consumo de Memoria RAM de la Matriz de Adyacencia densa: 80.7343 MB
Consumo de Memoria RAM del Dataframe limpio (lista de aristas): 1.1150 MB


# Veredicto

La Matriz de Adyacencia densa es aproximadamente 72.4 veces más pesada en memoria RAM que la Lista de Aristas.

##### ¿Por qué sucede esto?
La razón es que la matriz tiene un espacio para cada par posible de aeropuertos (más de 10 millones, porque hay 3,253), aunque en la realidad solo existen 9,743 vuelos directos. Por eso casi toda la matriz está llena de ceros que no sirven para nada. La lista, en cambio, anota solo las rutas que sí existen, así que gasta mucha menos memoria y es la mejor opción cuando hay pocas conexiones.
